# Behavior experiment (OLMo-1B, base model) — free-form generation + guardrail labeling

Same experiment as `behavior_eval_gemma-2-2b-it.ipynb`, but with **`allenai/OLMo-1B-hf`**, a *base* (non-instruction-tuned) model. We feed each raw statement from `mixed_dataset.csv` as a **raw completion** (no chat template, prompt ends with `The answer is:`) to see how a base model continues the text. Then we eyeball generations, classify each answer with an **LLM judge on OpenRouter** (`safe` / `harmful` / `gibberish`), and label safety with three guardrail models.

> **Run on a GPU host.** Generation uses vLLM; guardrail models load real weights. `shieldgemma-2b` is HF-gated — `huggingface-cli login` or `export HF_TOKEN=...` first. The LLM judge needs `export OPENROUTER_API_KEY=...`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from latent_alignment.data import load_statements
from latent_alignment.generate import (
    CHAT_TEMPLATE,
    DEFAULT_TEMPLATE,
    build_prompts,
    generate_continuations,
)
from latent_alignment.guardrails import GUARDRAILS
from latent_alignment.judge import DEFAULT_JUDGE_MODEL, judge_generations

In [ ]:
DATASET = "../data/polarity_probing/raw/mixed_dataset.csv"
GEN_MODEL = "allenai/OLMo-1B-hf"
USE_CHAT_TEMPLATE = False  # OLMo-1B is a base model -> raw completion.
SAMPLE_N = 30             # small slice first; set to None for the full dataset

# Instruct models get their native chat format + a question turn; base models get the raw
# "The answer is:" completion. Feeding each model in its own mode is intentional — an instruct
# model fed raw completion bypasses its safety tuning and behaves like a base model.
TEMPLATE = CHAT_TEMPLATE if USE_CHAT_TEMPLATE else DEFAULT_TEMPLATE

df = load_statements(DATASET)
if SAMPLE_N:
    df = df.groupby("label", group_keys=False).apply(
        lambda g: g.sample(min(len(g), SAMPLE_N // 2), random_state=0)
    ).reset_index(drop=True)
print(len(df), "statements |", "chat template" if USE_CHAT_TEMPLATE else "raw completion")
df.head()

## 1. Generate continuations (vLLM)

Base model → raw completion with `DEFAULT_TEMPLATE` (`The answer is:`). Instruct model → set `USE_CHAT_TEMPLATE = True` to wrap a question (`CHAT_TEMPLATE`) in the model's chat format, so safety tuning is actually engaged.

In [ ]:
statements = df["statement"].tolist()
# Display the user-facing prompt text; the chat-template wrapping (if any) is applied inside
# generate_continuations with the model's own tokenizer.
df["prompt"] = [TEMPLATE.format(statement=s) for s in statements]
df["generation"] = generate_continuations(
    statements,
    model_name=GEN_MODEL,
    template=TEMPLATE,
    use_chat_template=USE_CHAT_TEMPLATE,
    max_new_tokens=128,
    temperature=0.7,
)
df[["statement", "label", "generation"]].head(10)

## 2. Eyeball the generations

In [ ]:
with pd.option_context("display.max_colwidth", None):
    for _, row in df.head(20).iterrows():
        print(f"[label={row['label']}] {row['statement']}")
        print(f"  -> {row['generation'].strip()}\n")

## 3. LLM judge (3-class: safe / harmful / gibberish)

Instead of a regex stance heuristic, we ask another LLM on OpenRouter to read each statement +
generation and label the answer as `safe`, `harmful`, or `gibberish` (meaningless generation).
Set your key first: `export OPENROUTER_API_KEY=...`.

In [ ]:
verdicts = judge_generations(
    df["statement"].tolist(),
    df["generation"].tolist(),
    model=DEFAULT_JUDGE_MODEL,  # e.g. "openai/gpt-4o-mini" on OpenRouter
)
df["judge_label"] = [v["label"] for v in verdicts]
df["judge_reason"] = [v["reason"] for v in verdicts]
pd.crosstab(df["label"], df["judge_label"])

## 4. Guardrail labeling (3 models)

In [ ]:
prompts = df["statement"].tolist()
generations = df["generation"].tolist()

# Each classify_* loads its model fresh, so the previous guardrail is freed as we go.
for key, classify in GUARDRAILS.items():
    verdicts = classify(generations, prompts=prompts)
    df[f"{key}_unsafe"] = [v["unsafe"] for v in verdicts]
    df[f"{key}_score"] = [v["score"] for v in verdicts]

## 5. Summary

In [ ]:
unsafe_cols = [c for c in df.columns if c.endswith("_unsafe")]
summary = df.groupby("label")[unsafe_cols].mean()
summary

In [ ]:
df["n_unsafe_votes"] = df[unsafe_cols].sum(axis=1)
print(df["n_unsafe_votes"].value_counts().sort_index())
disagreements = df[df["n_unsafe_votes"].between(1, len(unsafe_cols) - 1)]
disagreements[["statement", "label", "judge_label", *unsafe_cols]]

In [ ]:
out = Path("../runs/behavior_olmo_1b")
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "behavior_results.csv", index=False)
print("wrote", out / "behavior_results.csv")